# Motores de entrenamiento en MONAI  (`monai.engines`)

MONAI ofrece motores (`SupervisedTrainer`, `SupervisedEvaluator`) basados en [PyTorch Ignite](https://pytorch-ignite.ai/) que encapsulan ese loop y permiten engancharle **handlers** reutilizables (logging, checkpoints, validación periódica, métricas, barra de progreso) sin reescribir el bucle.

Este notebook usa un ejemplo sintético simple para enfocarnos en la API de los engines.

In [ ]:
!pip install -q "monai[ignite,tqdm]"

import torch
from ignite.contrib.handlers import ProgressBar

import monai
from monai.data import DataLoader, Dataset
from monai.engines import SupervisedEvaluator, SupervisedTrainer
from monai.handlers import CheckpointSaver, MeanDice, StatsHandler, ValidationHandler, from_engine
from monai.transforms import Activationsd, AsDiscreted, Compose

monai.utils.set_determinism(seed=0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Datos sintéticos

Generamos imágenes 2D aleatorias de 1 canal con un círculo, y su máscara binaria correspondiente. Es solo para tener algo rápido de entrenar y poder observar la mecánica de los engines, no es representativo de un caso real.

In [ ]:
import numpy as np


def make_synthetic_sample(size=64):
    """Imagen con ruido + un círculo, y su máscara binaria correspondiente."""
    yy, xx = np.mgrid[0:size, 0:size]
    cx, cy = np.random.randint(20, size - 20, size=2).tolist()
    r = np.random.randint(8, 16)
    mask = ((xx - cx) ** 2 + (yy - cy) ** 2 <= r**2).astype(np.float32)
    image = mask * 0.6 + np.random.normal(0, 0.1, size=(size, size)).astype(np.float32)
    return image[None], mask[None]


class SyntheticDataset(Dataset):
    def __init__(self, n_samples=64):
        samples = [make_synthetic_sample() for _ in range(n_samples)]
        data = [{"image": img, "label": lbl} for img, lbl in samples]
        super().__init__(data=data, transform=None)


train_ds = SyntheticDataset(n_samples=64)
val_ds = SyntheticDataset(n_samples=16)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=4, num_workers=0)

In [ ]:
net = monai.networks.nets.UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64),
    strides=(2, 2),
).to(device)

loss_function = monai.losses.DiceLoss(sigmoid=True)
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

val_postprocessing = Compose(
    [
        Activationsd(keys="pred", sigmoid=True),
        AsDiscreted(keys="pred", threshold=0.5),
    ]
)

## `SupervisedEvaluator`

Encapsula el loop de validación: corre el modelo en modo `eval`, sin gradientes, y calcula métricas usando los handlers que le pasemos.

In [ ]:
evaluator = SupervisedEvaluator(
    device=device,
    val_data_loader=val_loader,
    network=net,
    postprocessing=val_postprocessing,
    key_val_metric={
        "val_mean_dice": MeanDice(include_background=True, output_transform=from_engine(["pred", "label"]))
    },
)

## `SupervisedTrainer`

Es el equivalente al loop `for epoch ... for batch ...` que escribimos a mano en el challenge, pero como objeto configurable. `ValidationHandler` conecta el evaluator para que corra cada tantas épocas, `CheckpointSaver` guarda el mejor modelo según la métrica clave, y `StatsHandler` imprime el progreso.

In [ ]:
max_epochs = 5

train_handlers = [
    ValidationHandler(validator=evaluator, interval=1, epoch_level=True),
    StatsHandler(tag_name="train_loss", output_transform=from_engine(["loss"], first=True)),
    CheckpointSaver(save_dir="./runs_engines", save_dict={"net": net}, save_key_metric=True),
    ProgressBar(),
]

trainer = SupervisedTrainer(
    device=device,
    max_epochs=max_epochs,
    train_data_loader=train_loader,
    network=net,
    optimizer=optimizer,
    loss_function=loss_function,
    train_handlers=train_handlers,
)

trainer.run()

## Loop manual vs. engines — equivalencias

| Loop tradicional (challenge) | Engine equivalente |
|---|---|
| `for epoch in range(max_epochs):` | `SupervisedTrainer(max_epochs=...)` |
| `for batch in train_loader: ... loss.backward(); optimizer.step()` | Lo hace `SupervisedTrainer` internamente |
| Bloque de validación manual con `torch.no_grad()` | `SupervisedEvaluator` + `ValidationHandler` |
| `print(...)` de la pérdida/métrica | `StatsHandler` |
| `torch.save(net.state_dict(), ...)` del mejor modelo | `CheckpointSaver(save_key_metric=True)` |
| Barra de progreso manual/`tqdm` | `ProgressBar` (de `ignite.contrib.handlers`) |

La ventaja de los engines aparece cuando el pipeline crece (múltiples métricas, checkpoints, early stopping, logging a TensorBoard, etc.): en vez de acumular lógica dentro del loop, se agregan handlers.